In [1]:
# was dedup-dependency
#from pathlib import Path
#from PIL import Image
#from hashlib import sha1
#from glob import iglob

# DEDUP

> Dedup was done successfully,  
> but due to Jupyter Notebook crash and save error,  
> All code with 3-hours of effors was gone.  
>  
> 1. Tried every method to recover original code,  
> but the last saved checkpoint was 3-hours ago.  
> 2. Tried to search Chrome browser cache binary,  
> browser cache was already removed.  
>  
> The result `index.txt` file is made successfully,  
> here, passing full-reimplement of indexing code.  
---
```
# Will consume some memory
class LocalIndex:
    def __init__(self):
        class CaptchaString: pass
        class FileHash: pass
        class FilePath: pass

        # Docs
        self._index: dict[CaptchaString: list[tuple[FilePath, FileHash]]] = dict()

    # This is temporal class,
    # Not optimized and structurized.
    def push(self, filepath:str) -> bool:

        # Will use checksum always.
        # Calculate here first.
        with open(filepath, "rb") as rawfile:
            file_hash_hex = sha1(rawfile, usedforsecurity=False).hexdigest()

        # Check file in index
        basename_wo_ext = Path(filepath).stem
        if exists := self._index.get(basename_wo_ext, None):
            for exist in exists:
                # Compare exist hash of file with same filename
                if (file_hash_hex == exist[1]):
                    return False;
            self._index[basename_wo_ext].append(
                (filepath, file_hash_hex)
            )
        else:
            self._index[basename_wo_ext] = [(filepath, file_hash_hex), ]

        return True;

local_index = LocalIndex()
# All non-PNG files were converted into PNG
for filepath in iglob("data/**/*.png"):
    if not local_index.push(filepath):
        print(filepath)
```
---

> According to previous experiment,  
> the biggest image size was (256x256).
>  
> And there are some grayscale and RGBA images,  
> here, converting every source image into  
> `(256, 256, 3)` sized images

---
### v0.3.0
> Max height was 256, but its all data was centered.  
> Hence, training in full height will waste computation power.  
> I decided to use (height=128, width=256) as data size.  
---

# Resize to unify shape and format

In [2]:
import torchvision.transforms as T
import numpy as np
import os, shutil, time, threading
from PIL import Image
from collections import Counter
from pathlib import Path

In [3]:
def get_dominant_corner_color(img: Image, _sample=0.05):
    img = np.array(img)
    w, h = img.shape[:2]
    channel = 1 if img.ndim == 2 else img.shape[2]

    pw = max(1, int(w * _sample))
    ph = max(1, int(h * _sample))

    corners = list()
    corners.append(img[0:ph, 0:pw]) # TL
    corners.append(img[0:ph, -pw:]) # TB
    corners.append(img[-ph:, 0:pw]) # BL
    corners.append(img[-ph:, -pw:]) # BR

    corners = np.concatenate([corner.reshape(-1, channel) for corner in corners], axis=0)

    pixels = [tuple(rgb) for rgb in corners]
    most_common = Counter(pixels).most_common(1)[0][0]

    return most_common;


In [4]:
def rgb_from_grayscale(img) -> Image:
    return img.convert("RGB")

In [5]:
def rgb_from_rgba(img) -> Image:
    bg_color = get_dominant_corner_color(img, _sample=0.1)
    bg = Image.new("RGB", img.size, bg_color)
    return Image.alpha_composite(bg, img).convert('RGB')

In [6]:
# max among `{ (40, 150, 3), (50, 200, 3), (50, 180), (50, 200, 4), (256, 256, 3) }`
MAX_H, MAX_W = (128, 256)

def reshape(img: Image) -> Image:
    w = img.width
    h = img.height

    mx = min(
        MAX_W / w,
        MAX_H / h,
    )

    hw = int( ( MAX_W - (w * mx) ) // 2 )
    hh = int( ( MAX_H - (h * mx) ) // 2 )

    unify = T.Compose([
        T.Pad(
            (hw, hh, hw, hh),
            fill=get_dominant_corner_color(img, _sample=0.1),
        ),
        T.Resize(
            (MAX_H,MAX_W)
        ),
    ])

    return unify(img);

In [7]:
class IndexLoader:
    def __init__(self, index_path="./data/index.txt"):
        # Loader
        with open(index_path, "r", encoding="utf-8") as index_txt:
            # SAFETY: `index.txt` will be a few `MB` level file. 
            # Directly read every data to memory. 
            self._index = index_txt.read()
        self._index = self._index.strip().split('\n')
        self._len = len(self._index)
        self._ptr = 0
        self._exhausted = False

        # Progress
        self._processed = 0
        self._out_index = dict()

        # Time Logger
        self._t0 = time.perf_counter();

    def __len__(self) -> int:
        return self._len

    def ready_workers(self, worker_n:int, out_dir:str):
        self._workers = [
            Worker(self, output=out_dir, _num=i)
            for i in range(worker_n)
        ]

    def halt(self):
        self._interrupted = True

    def run(self):
        self._interrupted = False

        self._threads = [
            threading.Thread(target=worker.run)
            for worker in self._workers
        ]

        for thread in self._threads:
            thread.start()

        for thread in self._threads:
            thread.join()

        print("IndexLoader: HALT OK")

    def get_batch(self, n) -> tuple[int, list[str]]:
        if self._exhausted:
            return 0, [];

        if (self._ptr+n) >= len(self):
            self._exhausted = True
            verified_n = len(self) - self._ptr
        else:
            verified_n = n

        batch = self._index[self._ptr:self._ptr+verified_n]
        self._ptr += verified_n;

        return verified_n, batch

    def get_sequence_of(self, name:str) -> int:
        if name in self._out_index:
            self._out_index[name] += 1;
            return self._out_index[name];

        self._out_index[name] = 0;
        return self._out_index[name];

    def report_progress(self, n, worker:int=-1):
        self._processed += n;
        print(f"Processed `{self._processed}` [ Last report from thread-`{worker:02}` ] ({time.perf_counter()-self._t0:.02f}s)", end="                \r")

        # Optional delay
        time.sleep(0.1)



class Worker:
    def __init__(self, loader:IndexLoader, output:str, _num=-1):
        # Worker label
        self._num = _num

        # Link Master
        self._loader = loader
        self._output = output

    def run(self):
        while not self._loader._interrupted:
            exit_code = self.process_batch(n=1000)
            if exit_code == 0:
                continue;
            break;
        print(f"Thread-`{self._num}`: HALT OK")

    def process_batch(self, n) -> int:
        verified_n, file_paths = self._loader.get_batch(n=n)

        for file_path in file_paths:

            # `file_path` is stripped by `IndexLoader`
            file_path = Path(file_path)
            basename_wo_ext = file_path.stem.split('.')[0]

            # Verify `file_path` is valid image
            try:
                Image.open(file_path).verify()
            except Exception as _:
                print(f"\nbroken: `{file_path}`)")
                file_path.unlink(missing_ok=True)
                continue;

            # Verify consumes Image
            # Reload
            img = Image.open(file_path)

            # Grayscale
            if len(img.size)==2 or img.size[2]==1:
                img = rgb_from_grayscale(img)
            # RGBA
            elif img.size[2]==4:
                img = rbg_from_rgba(img)

            img = reshape(img)

            seq = self._loader.get_sequence_of(basename_wo_ext)
            out_file_path = f"{self._output}{basename_wo_ext}.{seq}.png"

            img.save(out_file_path)

        if verified_n == 0:
            return 1;

        self.post_job(n=verified_n)
        return 0;

    def post_job(self, n:int):
        self._loader.report_progress(n=n, worker=self._num)


In [8]:
_idx = "./data/index.txt"
_dst = "./data/ready/"

try:
    print(f"Remove: `{_dst}`")
    shutil.rmtree(_dst)
except:
    print(f"Directory `{_dst}` does not exist.")

os.makedirs(_dst, exist_ok=False)

Remove: `./data/ready/`
Directory `./data/ready/` does not exist.


---
```
with open("./data/index.txt", "r", encoding="utf-8") as index_txt:
    _i = 0;
    _proc_per = 1000;
    _t0 = time.perf_counter()
    for filepath in index_txt.readlines():
        _i += 1;
        filepath = filepath.strip()
        basename_wo_ext = Path(filepath).stem

        if not _i % _proc_per:
            _dt = time.perf_counter() - _t0
            print(f"Processed `{_i}` [{filepath}] ({_dt:.02f}s)", end="                \r")

        # Verify valid
        try:
            Image.open(filepath).verify()
        except Exception as _:
            print(f"\nbroken: `{filepath}`)")
            Path(filepath).unlink(missing_ok=True)
            continue;

        # Verify consume Image
        img = Image.open(filepath)

        # Grayscale
        if len(img.size)==2 or img.size[2]==1:
            img = rgb_from_grayscale(img)
        # RGBA
        elif img.size[2]==4:
            img = rbg_from_rgba(img)

        img = reshape(img)

        # if there are more than 999 image for one captcha combination, 
        # this will result in overwrite the 100th result, 
        # but this will be an extreamly rare case. 
        for i in range(999):
            save_filepath = f"{_dst}{basename_wo_ext}.{i}.png"
            if Path(save_filepath).is_file():
                continue;
            break;
        img.save(save_filepath)
    print(f"Processed `{_i}` [{filepath}]", end="                \r")
```
---

In [9]:
loader = IndexLoader(
    index_path=_idx,
)

loader.ready_workers(
    10,
    out_dir=_dst,
)

In [10]:
try:
    loader.run()
except:
    loader.halt()

Processed `20000` [ Last report from thread-`05` ] (79.12s)                
broken: `data\archive (1)\Large_Captcha_Dataset\4q2wA.png`)
Thread-`6`: HALT OK[ Last report from thread-`06` ] (615.39s)                0` [ Last report from thread-`09` ] (497.22s)                
Thread-`5`: HALT OK[ Last report from thread-`05` ] (625.16s)                
Thread-`4`: HALT OK[ Last report from thread-`04` ] (629.41s)                
Thread-`1`: HALT OK[ Last report from thread-`01` ] (630.47s)                
Thread-`0`: HALT OK[ Last report from thread-`00` ] (630.99s)                
Thread-`2`: HALT OK[ Last report from thread-`02` ] (631.15s)                
Thread-`3`: HALT OK[ Last report from thread-`03` ] (632.93s)                
Thread-`8`: HALT OK[ Last report from thread-`08` ] (635.66s)                
Thread-`9`: HALT OK[ Last report from thread-`09` ] (635.87s)                
Thread-`7`: HALT OK[ Last report from thread-`07` ] (636.45s)                
IndexLoader: HALT OK
